# Mapeamento Eleitoral e Visualização Geoespacial com Folium
## Eleição Presidencial 2026 — São João del-Rei (MG)

**Universidade Federal de São João del-Rei (UFSJ) — DCOMP**
**Disciplina:** Ferramentas para Data Science — Prof.ª Carolina Xavier
**Link repo:** https://github.com/MarinaWang11/Eleicoes_2026_DS.git
**Link deploy:** eleicoes2026-saojoaodelrei.streamlit.app

### Objetivo
Analisar e visualizar a distribuição espacial dos votos para Presidente (1º e 2º turnos de 2026) por **local de votação / seção eleitoral** em **São João del-Rei**, município sede da UFSJ, usando `pandas`, `geopandas` e `folium`.

### Fontes de dados (TSE — Portal de Dados Abertos, IBGE)
| Dado | Arquivo original (MG) | Uso |
|---|---|---|
| Boletim de Urna 1º turno | `bweb_1t_MG_051020261403.csv` | votos por seção e candidato |
| Boletim de Urna 2º turno | `bweb_2t_MG_261020260742.csv` | votos por seção e candidato |
| Eleitorado por local de votação | `eleitorado_local_votacao_2026_MG.csv` | nome, bairro, endereço e **latitude/longitude** de cada local |
| Malha municipal (IBGE) | API `servicodados.ibge.gov.br` | contorno do município |

### Estrutura do projeto
```
Eleicao/
├── sjdr.ipynb              <- este notebook (tratamento + mapas + relatório)
├── app_sjdr.py             <- painel Streamlit (executa o notebook se faltar algo)
├── environment.yml         <- ambiente conda com versões exatas
├── dados_brutos/           <- recortes brutos do TSE/IBGE para São João del-Rei
├── dados_processados/      <- tabelas agregadas por local de votação
└── mapas_html/             <- mapas interativos exportados
```

### Como executar
```bash
conda env create -f environment.yml
conda activate eleicao_sjdr
streamlit run app_sjdr.py      
```

## Configuração do ambiente

In [ ]:
import json
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="Could not detect GDAL data files")

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import requests
from scipy.spatial import Voronoi
from shapely.geometry import Polygon
from IPython.display import display

print(f"python    {sys.version.split()[0]}")
print(f"pandas    {pd.__version__}")
print(f"geopandas {gpd.__version__}")
print(f"folium    {folium.__version__}")
print(f"json      {json.__version__} (biblioteca padrão do Python)")

COD_MUNICIPIO_SJDR = "52493"   # código TSE de São João del-Rei
COD_IBGE_SJDR = "3162500"      # código IBGE de São João del-Rei

DIR_TSE = Path(".")
ARQ_TSE = {
    "bweb_1t": DIR_TSE / "bweb_1t_MG_051020261403.csv",
    "bweb_2t": DIR_TSE / "bweb_2t_MG_261020260742.csv",
    "locais": DIR_TSE / "eleitorado_local_votacao_2026_MG.csv",
}

DIR_BRUTOS = Path("dados_brutos")
DIR_PROC = Path("dados_processados")
DIR_MAPAS = Path("mapas_html")
for d in (DIR_BRUTOS, DIR_PROC, DIR_MAPAS):
    d.mkdir(exist_ok=True)

ARQ_BRUTO = {k: DIR_BRUTOS / f"{k}_sjdr.csv" for k in ARQ_TSE}
ARQ_MALHA = DIR_BRUTOS / "malha_sjdr_ibge.geojson"

COR_CANDIDATO = {22: "#2166ac", 13: "#b2182b"}   # azul (PL) / vermelho (PT)

python    3.12.13
pandas    3.1.1
geopandas 1.2.0
folium    1.0.0
json      2.0.9 (biblioteca padrão do Python)


## 1. Obtenção dos dados e recorte bruto de São João del-Rei

In [ ]:
def recortar_sjdr(chave, filtro_extra=None):
    destino = ARQ_BRUTO[chave]
    
    if destino.exists():
        print(f"[ok] {destino} já existe — usando recorte salvo")
        return
        
    origem = ARQ_TSE[chave]
    
    if not origem.exists():
        raise FileNotFoundError(
            f"{destino} não existe e o arquivo original {origem} também não. "
            "Baixe-o em https://dadosabertos.tse.jus.br/ (ver README)."
        )
    pedacos = []
    
    for chunk in pd.read_csv(origem, sep=";", encoding="latin1", dtype=str, chunksize=500_000):
        filtro = chunk["CD_MUNICIPIO"] == COD_MUNICIPIO_SJDR
        if filtro_extra is not None:
            filtro &= filtro_extra(chunk)
        pedacos.append(chunk[filtro])
        
    df = pd.concat(pedacos, ignore_index=True)
    df.to_csv(destino, sep=";", index=False, encoding="utf-8")
    
    print(f"[novo] {destino}: {len(df):,} linhas recortadas de {origem}")

so_presidente = lambda c: c["CD_CARGO_PERGUNTA"] == "1"
recortar_sjdr("bweb_1t", so_presidente)
recortar_sjdr("bweb_2t", so_presidente)
recortar_sjdr("locais")

# Contorno oficial do município (IBGE)
if not ARQ_MALHA.exists():
    url = f"https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{COD_IBGE_SJDR}?formato=application/vnd.geo+json"
    ARQ_MALHA.write_text(json.dumps(requests.get(url, timeout=60).json()), encoding="utf-8")
gdf_mun = gpd.read_file(ARQ_MALHA).set_crs("EPSG:4326", allow_override=True)
poligono_municipio = gdf_mun.geometry.iloc[0]

for k, p in {**ARQ_BRUTO, "malha": ARQ_MALHA}.items():
    print(f"{p}: {p.stat().st_size / 1e6:.2f} MB")

[ok] dados_brutos\bweb_1t_sjdr.csv já existe — usando recorte salvo
[ok] dados_brutos\bweb_2t_sjdr.csv já existe — usando recorte salvo
[ok] dados_brutos\locais_sjdr.csv já existe — usando recorte salvo
dados_brutos\bweb_1t_sjdr.csv: 0.76 MB
dados_brutos\bweb_2t_sjdr.csv: 0.35 MB
dados_brutos\locais_sjdr.csv: 0.18 MB
dados_brutos\malha_sjdr_ibge.geojson: 0.02 MB


## 2. Limpeza, filtragem e unificação dos dados

### 2.1 Boletins de urna
* **Padronização das chaves:** `CD_MUNICIPIO` (5 dígitos), `NR_ZONA`, `NR_LOCAL_VOTACAO` e `NR_SECAO` (4 dígitos, com zeros à esquerda). 
* **Votos válidos:** `DS_TIPO_VOTAVEL == "Nominal"`.

In [ ]:
def padronizar_chaves(df):
    df = df.copy()
    df["CD_MUNICIPIO"] = df["CD_MUNICIPIO"].str.strip().str.zfill(5)
    for col in ["NR_ZONA", "NR_LOCAL_VOTACAO", "NR_SECAO"]:
        df[col] = df[col].str.strip().str.zfill(4)
    return df

def carregar_bu(chave):
    df = pd.read_csv(ARQ_BRUTO[chave], sep=";", dtype=str)
    df = padronizar_chaves(df)
    df["NR_VOTAVEL"] = df["NR_VOTAVEL"].astype(int)
    df["QT_VOTOS"] = df["QT_VOTOS"].astype(int)
    return df

bu_1t_completo = carregar_bu("bweb_1t")
bu_2t_completo = carregar_bu("bweb_2t")
bu_1t = bu_1t_completo[bu_1t_completo["DS_TIPO_VOTAVEL"] == "Nominal"].copy()
bu_2t = bu_2t_completo[bu_2t_completo["DS_TIPO_VOTAVEL"] == "Nominal"].copy()

def resumo_candidatos(bu):
    r = (bu.groupby(["NR_VOTAVEL", "NM_VOTAVEL", "SG_PARTIDO"], as_index=False)["QT_VOTOS"].sum()
           .sort_values("QT_VOTOS", ascending=False, ignore_index=True))
    r["PCT_VALIDOS"] = (r["QT_VOTOS"] / r["QT_VOTOS"].sum() * 100).round(2)
    return r

resumo_1t = resumo_candidatos(bu_1t)
resumo_2t = resumo_candidatos(bu_2t)

for t, bu in [("1º", bu_1t_completo), ("2º", bu_2t_completo)]:
    print(f"{t} turno: {bu['NR_SECAO'].nunique()} seções | {bu['NR_LOCAL_VOTACAO'].nunique()} locais | "
          f"zonas {sorted(bu['NR_ZONA'].unique())} | votos por tipo: "
          f"{bu.groupby('DS_TIPO_VOTAVEL')['QT_VOTOS'].sum().to_dict()}")

display(resumo_1t)
display(resumo_2t)

1º turno: 218 seções | 67 locais | zonas ['0328'] | votos por tipo: {'Branco': 1423, 'Nominal': 52332, 'Nulo': 1717}
2º turno: 218 seções | 67 locais | zonas ['0328'] | votos por tipo: {'Branco': 784, 'Nominal': 55421, 'Nulo': 1278}


,NR_VOTAVEL,NM_VOTAVEL,SG_PARTIDO,QT_VOTOS,PCT_VALIDOS
0,22,FLAVIO BOLSONARO,PL,25618,48.95
1,13,LULA,PT,21946,41.94
2,70,ESCRITOR AUGUSTO CURY,AVANTE,1858,3.55
3,14,RENAN SANTOS,MISSÃO,1157,2.21
4,55,RONALDO CAIADO,PSD,1102,2.11
5,30,ZEMA,NOVO,466,0.89
6,80,SAMARA,UP,76,0.15
7,16,HERTZ DIAS,PSTU,48,0.09
8,27,CLARIANA BARAO,DC,21,0.04
9,21,EDMILSON COSTA,PCB,17,0.03


,NR_VOTAVEL,NM_VOTAVEL,SG_PARTIDO,QT_VOTOS,PCT_VALIDOS
0,22,FLAVIO BOLSONARO,PL,27777,50.12
1,13,LULA,PT,27644,49.88


### 2.2 Locais de votação e coordenadas
O arquivo de eleitorado por local de votação tem uma linha por seção, com o local, o bairro, o endereço e a latitude/longitude. Trata a vírgula decimal e os valores ausentes do TSE (`-1`) e consolida uma linha por local de votação (zona + número do local).

In [ ]:
locais = padronizar_chaves(pd.read_csv(ARQ_BRUTO["locais"], sep=";", dtype=str))

for col in ["NR_LATITUDE", "NR_LONGITUDE"]:
    locais[col] = pd.to_numeric(locais[col].str.replace(",", ".", regex=False).str.strip(), errors="coerce")
    locais.loc[locais[col] == -1, col] = np.nan

CHAVE = ["CD_MUNICIPIO", "NR_ZONA", "NR_LOCAL_VOTACAO"]

locais_geo = (
    locais.dropna(subset=["NR_LATITUDE", "NR_LONGITUDE"])
          .groupby(CHAVE, as_index=False)
          .agg(NM_LOCAL_VOTACAO=("NM_LOCAL_VOTACAO", "first"),
               NM_BAIRRO=("NM_BAIRRO", "first"),
               DS_ENDERECO=("DS_ENDERECO", "first"),
               NR_LATITUDE=("NR_LATITUDE", "first"),
               NR_LONGITUDE=("NR_LONGITUDE", "first"))
)

print(f"Locais com coordenadas válidas: {len(locais_geo)} "
      f"(linhas sem coordenada descartadas: {locais['NR_LATITUDE'].isna().sum()})")

locais_geo.head()

Locais com coordenadas válidas: 72 (linhas sem coordenada descartadas: 0)


,CD_MUNICIPIO,NR_ZONA,NR_LOCAL_VOTACAO,NM_LOCAL_VOTACAO,NM_BAIRRO,DS_ENDERECO,NR_LATITUDE,NR_LONGITUDE
0,52493,0328,1023,COLEGIO NOSSA SENHORA DAS DORES,CENTRO,"RUA COMENDADOR BASTOS, 84",-21.136127,-44.257192
1,52493,0328,1040,CONSEVATÓRIO ESTADUAL DE MUSICA,CENTRO,"R. PADRE JOSÉ MARIA XAVIER, 164",-21.140247,-44.258927
2,52493,0328,1058,ESCOLA MUNICIPAL MARIA TERESA,CENTRO,"PRACA DOS EXPEDICIONARIOS, S/N",-21.137464,-44.258335
3,52493,0328,1066,ESCOLA MUNICIPAL MARIA TERESA,CENTRO,PRAÇA DOS EXPEDICIONÁRIOS S/N,-21.137464,-44.258335
4,52493,0328,1074,UFSJ - CAMPUS SANTO ANTONIO,CENTRO,"PRACA FREI ORLANDO, 170",-21.138325,-44.261110


### 2.3 Agregação dos votos por local de votação e junção com as coordenadas

In [ ]:
def votos_por_local(bu):
    piv = bu.pivot_table(index=CHAVE, columns="NR_VOTAVEL", values="QT_VOTOS", aggfunc="sum", fill_value=0)
    piv.columns = [f"votos_{c}" for c in piv.columns]
    secoes = bu.groupby(CHAVE)["NR_SECAO"].agg(
        LISTA_SECOES=lambda s: ", ".join(sorted(s.unique())), QTD_SECOES="nunique")
    return piv.join(secoes).reset_index()

def unir_coordenadas(bu, rotulo):
    vot = votos_por_local(bu)
    base = vot.merge(locais_geo, on=CHAVE, how="left", indicator=True)
    sem_geo = base[base["_merge"] == "left_only"]
    cols_v = [c for c in vot.columns if c.startswith("votos_")]
    print(f"{rotulo}: {len(vot)} locais no boletim | {len(sem_geo)} sem coordenada "
          f"({sem_geo[cols_v].sum().sum():.0f} votos válidos, "
          f"{sem_geo[cols_v].sum().sum() / vot[cols_v].sum().sum():.2%} do total)")
    return base[base["_merge"] == "both"].drop(columns="_merge"), sem_geo.drop(columns="_merge")

base_1t, sem_geo_1t = unir_coordenadas(bu_1t, "1º turno")
base_2t, sem_geo_2t = unir_coordenadas(bu_2t, "2º turno")
display(sem_geo_2t[CHAVE + ["LISTA_SECOES", "QTD_SECOES"] + [c for c in sem_geo_2t if c.startswith("votos_")]])

1º turno: 67 locais no boletim | 2 sem coordenada (527 votos válidos, 1.01% do total)
2º turno: 67 locais no boletim | 2 sem coordenada (557 votos válidos, 1.01% do total)


,CD_MUNICIPIO,NR_ZONA,NR_LOCAL_VOTACAO,LISTA_SECOES,QTD_SECOES,votos_13,votos_22
40,52493,0328,1619,0190,1,122,111
61,52493,0328,1856,0318,1,158,166


Os locais **1619** e **1856** aparecem nos boletins de urna mas não constam no cadastro de locais de votação do TSE, portanto não têm coordenada e ficam fora dos mapas (cerca de 1% dos votos válidos). Eles continuam incluídos nos totais municipais.

In [ ]:
def agrupar_por_coordenada(base):
    cols_v = [c for c in base.columns if c.startswith("votos_")]
    juntar = lambda s: " / ".join(dict.fromkeys(s))  # únicos, preservando a ordem
    agg = {c: "sum" for c in cols_v}
    agg.update(NR_ZONA=juntar, NR_LOCAL_VOTACAO=juntar, NM_LOCAL_VOTACAO=juntar,
               NM_BAIRRO="first", DS_ENDERECO="first", QTD_SECOES="sum",
               LISTA_SECOES=lambda s: ", ".join(sorted(", ".join(s).split(", "))))
    g = base.groupby(["NR_LATITUDE", "NR_LONGITUDE"], as_index=False).agg(agg)
    g["id_local"] = g["NR_LOCAL_VOTACAO"]
    return g

def calcular_indicadores(g, sufixo, nomes):
    cols_v = [c for c in g.columns if c.startswith("votos_")]
    g[f"total_validos_{sufixo}"] = g[cols_v].sum(axis=1)
    for c in cols_v:
        num = c.split("_")[1]
        g[f"pct_{num}_{sufixo}"] = (g[c] / g[f"total_validos_{sufixo}"] * 100).round(2)
    lider = g[cols_v].idxmax(axis=1).str.split("_").str[1].astype(int)
    g[f"lider_num_{sufixo}"] = lider
    g[f"lider_{sufixo}"] = lider.map(nomes)
    g[f"lider_pct_{sufixo}"] = [g.at[i, f"pct_{n}_{sufixo}"] for i, n in lider.items()]
    return g

NOMES = dict(zip(resumo_1t["NR_VOTAVEL"], resumo_1t["NM_VOTAVEL"]))
c1_num, c2_num = resumo_2t["NR_VOTAVEL"].iloc[:2].tolist()   # finalistas: 22 e 13
c1_nome, c2_nome = NOMES[c1_num], NOMES[c2_num]

locais_1t = calcular_indicadores(agrupar_por_coordenada(base_1t), "1t", NOMES)
locais_2t = calcular_indicadores(agrupar_por_coordenada(base_2t), "2t", NOMES)
locais_2t["saldo_pct"] = (locais_2t[f"pct_{c1_num}_2t"] - locais_2t[f"pct_{c2_num}_2t"]).round(2)
locais_2t.loc[locais_2t["saldo_pct"] == 0, "lider_2t"] = "EMPATE"

print(f"Feições no mapa: {len(locais_1t)} prédios (de {len(base_1t)} locais de votação)")
locais_2t[["NM_LOCAL_VOTACAO", "NM_BAIRRO", "QTD_SECOES", f"pct_{c1_num}_2t", f"pct_{c2_num}_2t", "saldo_pct", "lider_2t"]].head()

Feições no mapa: 61 prédios (de 65 locais de votação)


,NM_LOCAL_VOTACAO,NM_BAIRRO,QTD_SECOES,pct_22_2t,pct_13_2t,saldo_pct,lider_2t
0,CAPELA DA C0MUNIDADE DO ZUEIRA,POVOADO DO ZUEIRA,1,90.36,9.64,80.72,FLAVIO BOLSONARO
1,CAPELA DO VALO NOVO,POVOADO DO VALO NOVO,1,58.11,41.89,16.22,FLAVIO BOLSONARO
2,SALÃO DA CAPELA DE SÃO BENEDITO,POVOADO DA VENDINHA,1,53.06,46.94,6.12,FLAVIO BOLSONARO
3,SALÃO COMUNITARIO DO CAQUENDE,POVOADO DE CAQUENDE,1,33.33,66.67,-33.34,LULA
4,SALÃO PAROQUIAL DE CRUZEIRO DA BARRA,POVOADO DE CRUZEIRO DA BARRA,1,60.00,40.00,20.00,FLAVIO BOLSONARO


### 2.4 Geometria: áreas de influência
Para construir um mapa coroplético, cada local recebe a área do município mais próxima dele do que de qualquer outro local. Essa área aproxima a região de onde vêm os eleitores daquele local e é recortada pelo contorno oficial do IBGE.

In [ ]:
def gerar_voronoi(df):
    coords = df[["NR_LONGITUDE", "NR_LATITUDE"]].to_numpy()
    # 4 pontos distantes garantem que todas as regiões reais sejam finitas
    moldura = np.array([[-46.5, -23.5], [-42.0, -23.5], [-42.0, -19.0], [-46.5, -19.0]])
    vor = Voronoi(np.vstack([coords, moldura]))
    poligonos = []
    
    for i in range(len(coords)):
        regiao = vor.regions[vor.point_region[i]]
        poligonos.append(Polygon(vor.vertices[regiao]).intersection(poligono_municipio))
    return gpd.GeoDataFrame(df.copy(), geometry=poligonos, crs="EPSG:4326")

gdf_1t = gerar_voronoi(locais_1t)
gdf_2t = gerar_voronoi(locais_2t)
fora = (~gdf_1t.geometry.is_empty).sum()

print(f"{fora} de {len(gdf_1t)} polígonos não vazios dentro do município")

# Região de cada local: distância ao centro histórico
centro = gpd.GeoSeries.from_xy([-44.2610], [-21.1360], crs="EPSG:4326").to_crs(31983).iloc[0]
ROTULOS_REGIAO = ["Centro (< 1,5 km)", "Bairros urbanos (1,5–5 km)", "Distritos / zona rural (> 5 km)"]

def adicionar_regiao(df):
    pts = gpd.GeoSeries.from_xy(df["NR_LONGITUDE"], df["NR_LATITUDE"], crs="EPSG:4326").to_crs(31983)
    dist = (pts.distance(centro) / 1000).to_numpy()
    df["dist_centro_km"] = dist.round(2)
    df["regiao"] = pd.cut(dist, [0, 1.5, 5, 1000], labels=ROTULOS_REGIAO).astype(str)

for df in (gdf_1t, gdf_2t, locais_1t, locais_2t):
    adicionar_regiao(df)

61 de 61 polígonos não vazios dentro do município


In [ ]:
locais_1t.to_csv(DIR_PROC / "sjdr_locais_1t.csv", index=False)
locais_2t.to_csv(DIR_PROC / "sjdr_locais_2t.csv", index=False)
resumo_1t.to_csv(DIR_PROC / "sjdr_resumo_1t.csv", index=False)
resumo_2t.to_csv(DIR_PROC / "sjdr_resumo_2t.csv", index=False)

pd.concat([sem_geo_1t.assign(turno=1), sem_geo_2t.assign(turno=2)]).to_csv(DIR_PROC / "sjdr_locais_sem_coordenada.csv", index=False)

gdf_2t[["id_local", "geometry"]].to_file(DIR_PROC / "sjdr_areas_voronoi.geojson", driver="GeoJSON")

print(sorted(p.name for p in DIR_PROC.iterdir()))

['sjdr_areas_voronoi.geojson', 'sjdr_locais_1t.csv', 'sjdr_locais_2t.csv', 'sjdr_locais_sem_coordenada.csv', 'sjdr_resumo_1t.csv', 'sjdr_resumo_2t.csv']


## 3. Análise exploratória dos padrões de votação

In [ ]:
def soma_regiao(df, sufixo, nums):
    cols = [f"votos_{n}" for n in nums] + [f"total_validos_{sufixo}"]
    r = df.groupby("regiao")[cols].sum()
    out = pd.DataFrame({"locais": df.groupby("regiao").size(), "votos válidos": r[f"total_validos_{sufixo}"]})
    
    for n in nums:
        out[f"% {NOMES[n]}"] = (r[f"votos_{n}"] / r[f"total_validos_{sufixo}"] * 100).round(1)
    return out

print("1º turno por região")
display(soma_regiao(locais_1t, "1t", [c1_num, c2_num]))

print("2º turno por região")
display(soma_regiao(locais_2t, "2t", [c1_num, c2_num]))

print("Locais vencidos por cada candidato")
display(pd.DataFrame({"1º turno": locais_1t["lider_1t"].value_counts(),
                      "2º turno": locais_2t["lider_2t"].value_counts()}).fillna(0).astype(int))

1º turno por região


,locais,votos válidos,% FLAVIO BOLSONARO,% LULA
regiao,,,,
"Bairros urbanos (1,5–5 km)",19,24826,50.2,40.4
"Centro (< 1,5 km)",21,19405,47.1,44.1
Distritos / zona rural (> 5 km),21,7574,49.9,41.4


2º turno por região


,locais,votos válidos,% FLAVIO BOLSONARO,% LULA
regiao,,,,
"Bairros urbanos (1,5–5 km)",19,26407,51.6,48.4
"Centro (< 1,5 km)",21,20505,47.9,52.1
Distritos / zona rural (> 5 km),21,7952,50.8,49.2


Locais vencidos por cada candidato


,1º turno,2º turno
EMPATE,0,2
FLAVIO BOLSONARO,43,34
LULA,18,25


In [ ]:
cols_show = ["NM_LOCAL_VOTACAO", "NM_BAIRRO", "regiao", "total_validos_2t", "saldo_pct"]

print(f"Maiores vantagens de {c1_nome} no 2º turno (locais com 300+ votos)")
grandes = locais_2t[locais_2t["total_validos_2t"] >= 300]
display(grandes.nlargest(5, "saldo_pct")[cols_show])

print(f"Maiores vantagens de {c2_nome} no 2º turno (locais com 300+ votos)")
display(grandes.nsmallest(5, "saldo_pct")[cols_show])

comp = locais_1t[["id_local", "NM_LOCAL_VOTACAO", "NM_BAIRRO", "lider_1t", f"pct_{c1_num}_1t", f"pct_{c2_num}_1t"]].merge(
    locais_2t[["id_local", "lider_2t", f"pct_{c1_num}_2t", f"pct_{c2_num}_2t"]], on="id_local")
comp[f"ganho_{c1_num}"] = comp[f"pct_{c1_num}_2t"] - comp[f"pct_{c1_num}_1t"]
comp[f"ganho_{c2_num}"] = comp[f"pct_{c2_num}_2t"] - comp[f"pct_{c2_num}_1t"]

print(f"Ganho médio entre turnos (p.p.): {c1_nome} {comp[f'ganho_{c1_num}'].mean():.1f} | {c2_nome} {comp[f'ganho_{c2_num}'].mean():.1f}")

print(f"Ganho municipal (p.p.): {c1_nome} {resumo_2t.set_index('NR_VOTAVEL').at[c1_num,'PCT_VALIDOS'] - resumo_1t.set_index('NR_VOTAVEL').at[c1_num,'PCT_VALIDOS']:.2f} | "
      f"{c2_nome} {resumo_2t.set_index('NR_VOTAVEL').at[c2_num,'PCT_VALIDOS'] - resumo_1t.set_index('NR_VOTAVEL').at[c2_num,'PCT_VALIDOS']:.2f}")

viradas = comp[comp["lider_1t"] != comp["lider_2t"]]

print(f"Locais em que o líder mudou do 1º para o 2º turno: {len(viradas)}")
display(viradas[["NM_LOCAL_VOTACAO", "NM_BAIRRO", "lider_1t", "lider_2t", f"pct_{c1_num}_2t", f"pct_{c2_num}_2t"]])

Maiores vantagens de FLAVIO BOLSONARO no 2º turno (locais com 300+ votos)


,NM_LOCAL_VOTACAO,NM_BAIRRO,regiao,total_validos_2t,saldo_pct
60,GRUPO ESCOLAR MUNICIPAL PARADA DO GIAROLA,COLÔNIA DO GIAROLA,Distritos / zona rural (> 5 km),438,19.18
37,CENTRO DA PASTORAL DA PAROQUIA DE SÃO JUDAS TADEU,SÃO JUDAS TADEU,"Centro (< 1,5 km)",986,18.46
41,ESCOLA ESTADUAL GOVERNADOR MILTON CAMPOS,MATOSINHOS,"Bairros urbanos (1,5–5 km)",2324,18.16
30,ESCOLA MUNICIPAL MARIA TERESA,CENTRO,"Centro (< 1,5 km)",1154,17.50
46,SERVICO NACIONAL DE APRENDIZAGEM INDUSTRIAL SENAI,MATOSINHOS,"Bairros urbanos (1,5–5 km)",2104,17.20


Maiores vantagens de LULA no 2º turno (locais com 300+ votos)


,NM_LOCAL_VOTACAO,NM_BAIRRO,regiao,total_validos_2t,saldo_pct
34,SALÃO PAROQUIAL N SENHORA APARECIDA,ALTO DAS MERCES,"Centro (< 1,5 km)",507,-34.12
36,SALAO PAROQUIAL NOSSA SENHORA AUXILIADORA,SENHOR DOS MONTES,"Centro (< 1,5 km)",337,-33.54
14,ESCOLA ESTADUAL EVANDRO AVILA,DISTRITO DO RIO DAS MORTES,Distritos / zona rural (> 5 km),1289,-30.64
48,SALAO PAROQUIAL SAO GERALDO,SAO GERALDO,"Centro (< 1,5 km)",536,-27.24
22,ESCOLA MUNICIPAL DR. KLEBER VASQUES FILGUEIRAS,TEJUCO,"Bairros urbanos (1,5–5 km)",1656,-23.18


Ganho médio entre turnos (p.p.): FLAVIO BOLSONARO 1.3 | LULA 7.8
Ganho municipal (p.p.): FLAVIO BOLSONARO 1.17 | LULA 7.94
Locais em que o líder mudou do 1º para o 2º turno: 9


,NM_LOCAL_VOTACAO,NM_BAIRRO,lider_1t,lider_2t,pct_22_2t,pct_13_2t
8,ESCOLA MUNICIPAL DE EMBOABAS,ZONA RURAL,FLAVIO BOLSONARO,LULA,46.55,53.45
16,SALÃO PAROQUIAL DO FÉ,POVOADO DO FÉ,FLAVIO BOLSONARO,LULA,49.59,50.41
17,OBRAS SOCIAIS FE E ALEGRIA,TEJUCO,FLAVIO BOLSONARO,LULA,49.24,50.76
18,CENTRO SOCIAL SAO FRANCISCO,BONFIM,FLAVIO BOLSONARO,LULA,42.20,57.80
21,ESCOLA ESTADUAL CONEGO OSWALDO LUSTOSA,GUARDA-MOR,FLAVIO BOLSONARO,LULA,47.72,52.28
23,CENTRO EDUC. FREI SERÁFICO,SAO CAETANO,FLAVIO BOLSONARO,EMPATE,50.00,50.00
27,CESEC,TEJUCO,FLAVIO BOLSONARO,LULA,47.22,52.78
31,BANDA TEODORO DE FARIA,TEJUCO,FLAVIO BOLSONARO,LULA,49.54,50.46
35,UNIDADE REGIONAL DE SAÚDE (ANTIGO CENTRO DE SA...,CENTRO,FLAVIO BOLSONARO,EMPATE,50.00,50.00


## 4. Mapas coropléticos do 1º turno 

Os dois principais candidatos do 1º turno em São João del-Rei foram Flávio Bolsonaro (PL, 22) e Lula (PT, 13), que somaram 90,9% dos votos válidos. O terceiro colocado (Augusto Cury, 3,6%) não tem variação espacial suficiente.

In [ ]:
BINS_1T = [0, 35, 40, 45, 50, 55, 60, 100]

CSS_TOOLTIP = """
<style>
.foliumtooltip { padding: 0 !important; border: 0 !important; border-radius: 8px !important;
                 box-shadow: 0 4px 14px rgba(15,23,42,.35) !important; font-family: 'Segoe UI', Arial, sans-serif; }
.foliumtooltip table { border-collapse: collapse; margin: 0; min-width: 260px; }
.foliumtooltip th { background: #0f172a; color: #e2e8f0; text-align: left; font-weight: 600;
                    padding: 5px 8px; font-size: 11px; white-space: nowrap; }
.foliumtooltip td { padding: 5px 8px; font-size: 12px; color: #0f172a; border-bottom: 1px solid #e2e8f0; }
.badge { color: #fff; padding: 1px 6px; border-radius: 4px; font-weight: 600; }
.barra { background: #e2e8f0; border-radius: 3px; height: 8px; width: 140px; margin-top: 3px; }
.barra > div { height: 8px; border-radius: 3px; }
</style>
"""

def badge(nome, num, pct):
    cor = COR_CANDIDATO.get(num, "#475569")
    return f'<span class="badge" style="background:{cor}">{nome}</span> {pct:.1f}%'

def barra(pct, cor):
    return f'<b>{pct:.1f}%</b><div class="barra"><div style="width:{min(pct,100):.0f}%;background:{cor}"></div></div>'

def mapa_base():
    m = folium.Map(location=[-21.13, -44.26], zoom_start=13, tiles=None, control_scale=True)
    # Base cinza neutra (padrão) + OpenStreetMap 
    folium.TileLayer("Esri.WorldGrayCanvas", name="Mapa base cinza (Esri)").add_to(m)
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
    folium.GeoJson(gdf_mun, name="Limite municipal (IBGE)",
                   style_function=lambda f: {"fill": False, "color": "#334155", "weight": 2, "dashArray": "5 5"}).add_to(m)

    minx, miny, maxx, maxy = gdf_mun.total_bounds
    m.fit_bounds([[miny, minx], [maxy, maxx]], padding=(10, 10))
    m.get_root().header.add_child(folium.Element(CSS_TOOLTIP))
    return m

def criar_mapa_1t(num, paleta, arquivo):
    nome, cor = NOMES[num], COR_CANDIDATO[num]
    g = gdf_1t.copy()
    g["zona_secoes"] = "Zona " + g["NR_ZONA"] + " · Local " + g["NR_LOCAL_VOTACAO"] + " · Seções " + g["LISTA_SECOES"]
    g["lider_html"] = [badge(NOMES[n], n, p) for n, p in zip(g["lider_num_1t"], g["lider_pct_1t"])]
    g["cand_html"] = [barra(p, cor) + f'<small>{int(v)} de {int(t)} votos válidos</small>'
                      for p, v, t in zip(g[f"pct_{num}_1t"], g[f"votos_{num}"], g["total_validos_1t"])]

    m = mapa_base()
    
    ch = folium.Choropleth(
        geo_data=g.to_json(), data=g, columns=["id_local", f"pct_{num}_1t"],
        key_on="feature.properties.id_local", bins=BINS_1T, fill_color=paleta,
        fill_opacity=0.65, line_color="#ffffff", line_weight=1, line_opacity=0.9,
        nan_fill_color="#cbd5e1", highlight=True,
        name=f"% de votos válidos — {nome}",
        legend_name=f"1º turno — % dos votos válidos de {nome} ({num}) por local de votação",
    ).add_to(m)
    
    ch.geojson.add_child(folium.GeoJsonTooltip(
        fields=["NM_LOCAL_VOTACAO", "NM_BAIRRO", "zona_secoes", "lider_html", "cand_html"],
        aliases=["Local de votação", "Bairro", "Zona / seções", "Líder no local", nome],
        sticky=True))

    pontos = folium.FeatureGroup(name="Locais de votação (tamanho ∝ votos)")
    
    for _, r in g.iterrows():
        folium.CircleMarker([r["NR_LATITUDE"], r["NR_LONGITUDE"]], radius=2 + np.sqrt(r["total_validos_1t"]) / 6,
                            color="#0f172a", weight=1, fill=True, fill_color=cor, fill_opacity=0.9,
                            tooltip=f"{r['NM_LOCAL_VOTACAO']}: {r[f'pct_{num}_1t']:.1f}% {nome}").add_to(pontos)
    pontos.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    m.save(arquivo)
    return m

mapa_1t_c1 = criar_mapa_1t(c1_num, "YlGnBu", DIR_MAPAS / "mapa_1t_cand1.html")
mapa_1t_c2 = criar_mapa_1t(c2_num, "Reds", DIR_MAPAS / "mapa_1t_cand2.html")

print("Mapas do 1º turno salvos em", DIR_MAPAS)

Mapas do 1º turno salvos em mapas_html


### 4.1 Mapa 1 — 1º turno: Flávio Bolsonaro (PL, 22)

In [ ]:
mapa_1t_c1

**O fato.**
Flávio Bolsonaro venceu o 1º turno em São João del-Rei com 48,95% dos votos válidos (25.618 votos), sem atingir a maioria absoluta. O apoio é amplo: ele lidera em 43 dos 61 prédios de votação e passa de 50% em 29 deles, mas o mapa mostra predomínio de tons médios (45–55% em 27 prédios), o que indica um voto distribuído pela cidade.

* As áreas mais escuras ficam nos bairros urbanos fora do núcleo central (Matosinhos, Jardim Paulo Campos, COHAB), com mediana de 52,6% por local, e em povoados e colônias rurais (mediana 54,3%). Entre os locais grandes, os maiores percentuais estão na E.E. Governador Milton Campos (58,0%, 2.207 votos) e no SENAI (55,7%), ambos em Matosinhos, e no São Judas Tadeu (59,4%).
* Os tons claros se concentram em uma mancha no entorno do centro histórico (Alto das Mercês, Senhor dos Montes, São Geraldo), onde a mediana cai para 44,3%. São os mesmos lugares em que o mapa de Lula fica mais escuro.
* Valores extremos (8,8% e 85,5%) aparecem em povoados com uma única seção e poucas dezenas de eleitores. 

**O design.**
* Paleta sequencial `YlGnBu` (amarelo → verde → azul): remete às cores usadas pela campanha (verde e amarelo) e ao azul do PL, e termina no mesmo azul que representa o candidato no mapa do 2º turno. Por ser sequencial, a cor mais escura indica o maior percentual.
* 7 classes com limites fixos (0–35, 35–40, …, 55–60, 60–100%): a maior parte dos locais fica entre 35% e 60%, então classes de 5 p.p. nessa faixa mostram as diferenças. As duas classes extremas, mais largas, absorvem os poucos locais rurais com valores atípicos. São os mesmos limites do mapa de Lula, o que permite comparar os dois mapas.
* Opacidade 0,65 sobre um mapa base cinza neutro: sem cores próprias, ele não compete com a paleta, e as ruas e os nomes dos bairros continuam legíveis sob a camada de cor. As bordas brancas separam polígonos vizinhos sem pesar visualmente.
* Enquadramento automático: `fit_bounds` com os limites do município (IBGE) faz o mapa abrir mostrando todo o território, incluindo distritos e zona rural, sem precisar arrastar ou dar zoom.
* Componentes Folium: `Choropleth` (camada de cor e legenda), `GeoJsonTooltip` com HTML/CSS (líder com a cor do partido e barra de percentual), `CircleMarker` com raio proporcional à raiz dos votos válidos e `LayerControl` para ligar ou desligar os pontos e o limite municipal.

### 4.2 Mapa 2 — 1º turno: Lula (PT, 13)

In [ ]:
mapa_1t_c2

**O fato.**
Lula teve 41,94% dos votos válidos no 1º turno (21.946 votos), 7 p.p. atrás de Flávio. O mapa é o negativo do mapa anterior: Lula vence em 18 dos 61 prédios e passa de 50% em 10.

* A área mais escura é um arco de bairros populares próximos ao centro, nas encostas: Senhor dos Montes (60,0%), Alto das Mercês (59,0%) e São Geraldo (55,0%). Juntos, os locais a menos de 1,5 km do centro deram 44,1% a Lula, seu melhor resultado entre as três regiões.
* Fora do centro, se destacam o Distrito do Rio das Mortes (E.E. Evandro Ávila, 56,7% com 1.239 votos), o Tejuco (E.M. Dr. Kleber Vasques, 53,4%) e o Campus Dom Bosco da UFSJ (52,9% com 1.765 votos), que reúne muitos eleitores ligados à universidade.
* Nos bairros de Matosinhos, Jardim Paulo Campos e nas colônias rurais, o vermelho é claro (33–36%), o que coincide com as áreas fortes de Flávio.
* Os candidatos de terceira via (Cury 3,6%, Renan Santos 2,2%, Caiado 2,1%) tiveram votação homogênea, sem nenhum reduto espacial. Esses ~9% de votos ficaram em disputa para o 2º turno.

**O design.**
* Paleta sequencial `Reds`: o vermelho é a cor do PT e é o mesmo tom (`#b2182b`) que representa Lula no mapa divergente do 2º turno. Por ser monocromática, a intensidade da cor equivale diretamente ao percentual.
* Mesmas 7 classes e mesma opacidade (0,65) do mapa 1**, para que uma mesma cor intermediária signifique o mesmo percentual nos dois mapas. A diferença entre eles está apenas na cor.
* Tooltip, círculos proporcionais e `LayerControl` iguais aos do mapa 1, mostrando o líder do local (que pode ser o outro candidato) e a barra de percentual de Lula.

## 5. Mapa do 2º turno — comparação espacial dos finalistas

Valores positivos indicam vitória de Flávio (azul) e negativos vitória de Lula (vermelho). O zero, que representa o empate, é o ponto central da paleta divergente.

In [ ]:
BINS_2T = [-100, -30, -15, -5, 0, 5, 15, 30, 100]

g = gdf_2t.copy()
g["zona_secoes"] = "Zona " + g["NR_ZONA"] + " · Local " + g["NR_LOCAL_VOTACAO"] + " (" + g["QTD_SECOES"].astype(str) + " seções)"
g["lider_html"] = [("<b>EMPATE</b>" if s == 0 else badge(NOMES[n], n, p))
                   for n, p, s in zip(g["lider_num_2t"], g["lider_pct_2t"], g["saldo_pct"])]
g["c1_html"] = [barra(p, COR_CANDIDATO[c1_num]) + f"<small>{int(v)} votos</small>" for p, v in zip(g[f"pct_{c1_num}_2t"], g[f"votos_{c1_num}"])]
g["c2_html"] = [barra(p, COR_CANDIDATO[c2_num]) + f"<small>{int(v)} votos</small>" for p, v in zip(g[f"pct_{c2_num}_2t"], g[f"votos_{c2_num}"])]
g["saldo_txt"] = [f"{abs(s):.1f} p.p. para {NOMES[c1_num] if s > 0 else NOMES[c2_num]}" if s != 0 else "0" for s in g["saldo_pct"]]

mapa_2t = mapa_base()
ch2 = folium.Choropleth(
    geo_data=g.to_json(), data=g, columns=["id_local", "saldo_pct"],
    key_on="feature.properties.id_local", bins=BINS_2T, fill_color="RdBu",
    fill_opacity=0.7, line_color="#ffffff", line_weight=1, line_opacity=0.9, highlight=True,
    name="Saldo Flávio − Lula (áreas de influência)",
    legend_name=f"2º turno — saldo em p.p. (% {c1_nome} − % {c2_nome}): azul = {c1_nome}, vermelho = {c2_nome}",
).add_to(mapa_2t)

ch2.geojson.add_child(folium.GeoJsonTooltip(
    fields=["NM_LOCAL_VOTACAO", "NM_BAIRRO", "zona_secoes", "lider_html", "c1_html", "c2_html", "saldo_txt"],
    aliases=["Local de votação", "Bairro", "Zona / seções", "Líder no local", c1_nome, c2_nome, "Vantagem"],
    sticky=True))

pontos = folium.FeatureGroup(name="Locais de votação (tamanho ∝ votos, cor = vencedor)")

for _, r in g.iterrows():
    cor = "#64748b" if r["saldo_pct"] == 0 else COR_CANDIDATO[c1_num if r["saldo_pct"] > 0 else c2_num]
    popup = f"""
    <div style="font-family:'Segoe UI',Arial;width:250px">
      <h4 style="margin:0 0 4px;color:#0f172a">{r['NM_LOCAL_VOTACAO']}</h4>
      <small style="color:#475569"><b>Bairro:</b> {r['NM_BAIRRO']}<br><b>Seções:</b> {r['LISTA_SECOES']}</small>
      <hr style="margin:6px 0">
      <span style="color:{COR_CANDIDATO[c1_num]}"><b>{c1_nome}:</b> {int(r[f'votos_{c1_num}'])} ({r[f'pct_{c1_num}_2t']:.1f}%)</span><br>
      <span style="color:{COR_CANDIDATO[c2_num]}"><b>{c2_nome}:</b> {int(r[f'votos_{c2_num}'])} ({r[f'pct_{c2_num}_2t']:.1f}%)</span>
    </div>"""
    
    folium.CircleMarker([r["NR_LATITUDE"], r["NR_LONGITUDE"]], radius=2 + np.sqrt(r["total_validos_2t"]) / 6,
                        color="#0f172a", weight=1, fill=True, fill_color=cor, fill_opacity=0.9,
                        popup=folium.Popup(popup, max_width=270),
                        tooltip=f"{r['NM_LOCAL_VOTACAO']} — {r['lider_2t']}").add_to(pontos)
    
pontos.add_to(mapa_2t)
folium.LayerControl(collapsed=False).add_to(mapa_2t)
mapa_2t.save(DIR_MAPAS / "mapa_2t_comparativo.html")
mapa_2t

**O fato.**
O 2º turno foi mais disputado: Flávio Bolsonaro teve 27.777 votos (50,12%) contra 27.644 de Lula (49,88%), uma diferença de apenas 133 votos. O mapa divergente mostra uma cidade dividida em blocos:

* Lula recuperou quase todo o eleitorado dos candidatos eliminados: cresceu 7,9 p.p. em relação ao 1º turno, contra 1,2 p.p. de Flávio, e cresceu em 60 dos 61 prédios (média de +7,8 p.p. por local). Por isso o número de prédios vencidos por Lula passou de 18 para 25, Flávio ficou com 34 e houve 2 empates exatos (Centro Educ. Frei Seráfico e a Unidade Regional de Saúde).
* 9 locais mudaram de lado entre os turnos, a maioria onde Flávio havia vencido por pouco: Tejuco (Obras Sociais Fé e Alegria, CESEC, Banda Teodoro de Faria), Bonfim (Centro Social São Francisco), Guarda-Mor, Emboabas e Povoado do Fé. No mapa eles aparecem em vermelho claro* ou seja, são vitórias apertadas, de 0 a 5 p.p.
* Vermelho forte (vantagem > 15 p.p. para Lula): Alto das Mercês (+34 p.p.), Senhor dos Montes (+34), Distrito do Rio das Mortes (+31), São Geraldo (+27) e Tejuco/Kleber Vasques (+23). Com esses locais, o centro e seu entorno (< 1,5 km) deram 52,1% a Lula.
* Azul forte (vantagem > 15 p.p. para Flávio): Matosinhos (Milton Campos +18 p.p. e SENAI +17, ambos com mais de 2 mil votos), São Judas Tadeu (+18), Escola Maria Teresa (+18) e Colônia do Giarola (+19). Os bairros urbanos fora do centro (51,6%) e os distritos e a zona rural (50,8%) decidiram a eleição para Flávio.
* Os 2 locais sem coordenada (1619 e 1856, 557 votos) deram 277 votos a Flávio e 280 a Lula e não mudariam o resultado.

**O design.**
* Paleta divergente `RdBu` centrada no zero: vermelho para Lula (PT) e azul para Flávio (PL), as mesmas cores dos mapas do 1º turno. A intensidade indica o tamanho da vantagem e os tons claros indicam disputa equilibrada. Os limites das classes foram definidos manualmente e são simétricos em torno de 0: −100, −30, −15, −5, 0, 5, 15, 30, 100. Com as classes automáticas do folium (6 faixas iguais entre o mínimo −70 e o máximo +81), o zero ficaria dentro de uma faixa e locais vencidos por candidatos diferentes teriam a mesma cor.
* 8 classes (4 para cada lado): separam disputa apertada (até 5 p.p.), vantagem clara (5–15), forte (15–30) e esmagadora (> 30).
* Opacidade 0,7, um pouco maior que no 1º turno.
* Componentes Folium: `Choropleth` com o saldo; `GeoJsonTooltip` em HTML/CSS com o líder, as barras dos dois finalistas e a vantagem em p.p.; `FeatureGroup` com `CircleMarker` proporcionais aos votos, coloridos pelo vencedor e com `Popup` detalhado; `LayerControl` para alternar entre as áreas de influência e os pontos exatos do TSE.